In [0]:
# =============================================================================
# BRONZE LAYER — Road Restrictions Ingestion
# Toronto Chronic Congestion Root-Cause Intelligence Platform
#
# What this notebook does:
#   - Reads the Road Restrictions CSV file from the /data folder
#   - Skips the first row (it is a title: "Current road restrictions" — not data)
#   - Row 2 is the real column headers — Row 3 onward is real data
#   - Adds an ingestion_timestamp column
#   - Writes the data as a raw Bronze Delta table — NO cleaning, NO filtering
#
# Table created:
#   - bronze.road_restrictions
#
# Important notes about this dataset:
#   - Covers BOTH construction AND special events (same CSV file, same table)
#   - SpecialEvent column = "Yes" for events, "No" for construction
#   - Timestamps (CreatedTime, LastUpdated, StartTime, EndTime) are Unix milliseconds
#     e.g. 1778268992000 — we will convert these to readable dates in Silver
#   - Some columns contain embedded JSON text — we store them as-is in Bronze
#
# Run this manually once to create the initial table.
# After that, the Databricks Workflow will call it nightly.
# =============================================================================


# -----------------------------------------------------------------------------
# STEP 1 — Import libraries
# -----------------------------------------------------------------------------
# pandas     → We use it to read the CSV while skipping the title row
# pyspark    → Converts the pandas table into a Spark DataFrame for Delta
# F          → PySpark functions — we use current_timestamp() to stamp load time

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


# -----------------------------------------------------------------------------
# STEP 2 — Define the file path
# -----------------------------------------------------------------------------
# Update the email below to match YOUR Databricks account email

REPO_BASE = "/Workspace/Repos/panditadvait75@gmail.com/toronto-traffic-pipeline"

RESTRICTIONS_PATH = f"{REPO_BASE}/data/road_restrictions.csv"

# NOTE: Make sure you rename the downloaded file to "road_restrictions.csv"
# before uploading it to your GitHub /data folder.
# The original download name looks like: 1778775933176_v3.csv


# -----------------------------------------------------------------------------
# STEP 3 — Read the CSV file using pandas, skipping the title row
# -----------------------------------------------------------------------------
# Why pandas here instead of Spark directly?
#
# The CSV has a quirk: Row 1 is a plain text title ("Current road restrictions")
# and Row 2 is the actual column headers.
#
# Spark's CSV reader always treats the FIRST row as the header.
# So if we use Spark directly, it will try to make "Current road restrictions"
# the column name — which is wrong.
#
# pandas gives us a simple fix:
#   skiprows=1        → Skip row 1 (the title)
#   header=0          → Treat the next row (original row 2) as column names
#   dtype=str         → Read EVERY column as text — no type guessing
#   keep_default_na=False → Empty cells stay as empty string, not NaN

print("Reading Road Restrictions CSV file...")

pdf = pd.read_csv(
    RESTRICTIONS_PATH,
    skiprows=1,
    header=0,
    dtype=str,
    keep_default_na=False
)

print(f"Row count from file: {len(pdf)}")
print(f"Columns found: {list(pdf.columns)}")


# -----------------------------------------------------------------------------
# STEP 4 — Convert pandas DataFrame to Spark DataFrame
# -----------------------------------------------------------------------------
# spark.createDataFrame(pdf) converts the pandas table into a Spark DataFrame.
# Because we read everything as dtype=str, all columns arrive as StringType
# in Spark — exactly what we want for Bronze (raw, no type assumptions).

print("Converting to Spark DataFrame...")

df = spark.createDataFrame(pdf)


# -----------------------------------------------------------------------------
# STEP 5 — Add ingestion_timestamp
# -----------------------------------------------------------------------------
# Stamp every row with the exact time this notebook loaded the data.

df = df.withColumn("ingestion_timestamp", F.current_timestamp())

print("Schema:")
df.printSchema()


# -----------------------------------------------------------------------------
# STEP 6 — Write to Bronze Delta table
# -----------------------------------------------------------------------------
# mode("overwrite") → Replace the entire table each nightly run
#
# Why overwrite?
# The city publishes the CURRENT active list of restrictions each time.
# Records get added when construction starts, and disappear when it ends.
# We overwrite Bronze with the fresh full snapshot nightly.
#
# CDC logic (detecting what CHANGED between yesterday and today) is handled
# in Phase 4 Silver layer — not here. Bronze just stores the raw snapshot.

print("Writing to Bronze Delta table...")

(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.road_restrictions")
)

print("bronze.road_restrictions written successfully.")


# -----------------------------------------------------------------------------
# STEP 7 — Validation
# -----------------------------------------------------------------------------

print("\n--- VALIDATION ---")

row_count = spark.sql("SELECT COUNT(*) as row_count FROM bronze.road_restrictions").collect()[0]["row_count"]
print(f"bronze.road_restrictions row count: {row_count}")

print("\nTable preview (first 5 rows):")
spark.sql("SELECT * FROM bronze.road_restrictions LIMIT 5").display()

# Quick check — confirm both construction and event records exist
print("\nSpecialEvent value breakdown:")
spark.sql("""
    SELECT SpecialEvent, COUNT(*) as count
    FROM bronze.road_restrictions
    GROUP BY SpecialEvent
    ORDER BY count DESC
""").display()

print("\nBronze ingestion complete for Dataset 2 — Road Restrictions.")